In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="02-visual-search",
    notebook_name="03_ann_search_and_serving.ipynb"
)

# ANN Search & Serving Pipeline Deep Dive

## What This Notebook Covers

In Notebook 01 we designed the full system. In Notebook 02 we built the embedding model. Now we tackle the hardest engineering problem in visual search: **how do you search 200 billion embeddings in under 100 milliseconds?**

Think of it like this: you have a library with 200 billion books. Finding the most similar book by reading every one would take years. You need a smarter indexing system — that is exactly what Approximate Nearest Neighbor (ANN) search provides.

This notebook covers:

1. Exact nearest neighbor search — why brute force fails at scale
2. Tree-based ANN: KD-trees with scipy, Annoy overview
3. LSH (Locality-Sensitive Hashing) — implement from scratch
4. IVF (Inverted File Index) — clustering-based ANN
5. FAISS — industry-standard ANN library (demo with real code)
6. Product Quantization — memory optimization
7. Complete prediction pipeline: query -> embed -> ANN -> re-rank
8. Complete indexing pipeline: batch embed -> build/update index
9. Re-ranking service: business rules + filters
10. Scale analysis: how many GPUs for 200B images
11. Interview cheat sheet

---

## 1. Exact Nearest Neighbor Search (Brute Force)

### The Analogy

Imagine you lost your dog and you have a photo of her. The "brute force" approach is to show that photo to every single person in New York City (8 million people), one by one, asking "have you seen this dog?" It will definitely work — but it will take forever.

Exact NN does exactly this: it computes the similarity between the query embedding and **every single embedding** in the database.

### Complexity

```
Time complexity:  O(N × D)
    N = number of database vectors (200 billion)
    D = embedding dimension (256)

For 200B images × 256 dims = 51.2 trillion multiply-add operations per query
At 10 TFLOPS (GPU): ~5 seconds per query
```

That is completely unacceptable for a production system. We need sub-100ms latency.

### When Exact NN Is Fine

| Scale | Approach | Why |
|-------|----------|-----|
| < 10,000 images | Brute force | Fast enough, no complexity |
| < 1 million | Approximate is overkill | Brute force fits in RAM, < 1ms |
| > 10 million | Must use ANN | Brute force too slow |
| > 1 billion | ANN + distributed | Cannot fit in one machine |

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ======================================================================
# Exact Nearest Neighbor Search: Implementation and Benchmarking
#
# We benchmark across different N values to show the O(N*D) scaling.
# ======================================================================

def exact_nn_search(query, index, k=10):
    """
    Brute-force nearest neighbor search.
    
    Computes cosine similarity between query and every vector in the index.
    Returns the k most similar indices and their similarity scores.
    
    Time complexity: O(N * D)
    Space complexity: O(N * D) for the index
    """
    # Cosine similarity via dot product (embeddings are L2-normalized)
    similarities = index @ query  # (N,)
    
    # Get top-k indices (argsort returns ascending, so take last k)
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    top_k_sim = similarities[top_k_idx]
    
    return top_k_idx, top_k_sim


embedding_dim = 256

# Benchmark at different scales
scales = [1_000, 10_000, 100_000, 1_000_000]
search_times_ms = []

print("Exact NN Search Benchmark")
print(f"{'N (images)':>15} {'Time (ms)':>12} {'Projected @ 200B':>20}")
print("-" * 50)

for N in scales:
    # Create random L2-normalized embeddings
    index = np.random.randn(N, embedding_dim).astype(np.float32)
    norms = np.linalg.norm(index, axis=1, keepdims=True)
    index = index / norms
    
    # Create a random query
    query = np.random.randn(embedding_dim).astype(np.float32)
    query = query / np.linalg.norm(query)
    
    # Benchmark
    n_trials = 5
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        top_idx, top_sim = exact_nn_search(query, index, k=10)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    
    avg_ms = np.mean(times)
    search_times_ms.append(avg_ms)
    
    # Project to 200B
    projected_s = avg_ms / 1000 * (200_000_000_000 / N)
    
    print(f"{N:>15,} {avg_ms:>12.2f} {projected_s:>18.1f}s")

print()
print("At 200 billion images, brute force would take hundreds to thousands of seconds.")
print("We need Approximate Nearest Neighbor (ANN) to get sub-second results.")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scales, search_times_ms, 'o-', color='#E53935', linewidth=2.5, markersize=8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Indexed Images (N)', fontsize=13)
ax.set_ylabel('Search Time (ms)', fontsize=13)
ax.set_title('Exact NN Search: O(N×D) Scaling', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)

# Target latency line
ax.axhline(100, color='#4CAF50', linestyle='--', linewidth=2, label='Target: 100ms latency')
ax.legend(fontsize=11)

ax.text(0.02, 0.95, f'Embedding dim D = {embedding_dim}\nLinear growth → unacceptable at 200B scale',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.9))

plt.tight_layout()
plt.show()

## 2. Tree-Based ANN: KD-Trees and Annoy

### The Analogy

Instead of asking every person in New York where your dog is, you organize the city into neighborhoods. First you ask "is she in Manhattan or Brooklyn?" Once you know the borough, you only search that borough. This is what tree-based ANN does — it partitions the embedding space into regions and only searches the relevant region.

### KD-Trees

A **KD-tree** (K-Dimensional tree) recursively splits the embedding space along alternating dimensions, creating a binary tree. To find nearest neighbors:
1. Traverse the tree to find the leaf node closest to the query
2. Search only that subtree (and backtrack if needed)

**Time complexity:** O(D log N) on average — much better than O(N×D)\!

**The catch:** KD-trees work well in low dimensions (D < 20) but suffer from the **curse of dimensionality** in high dimensions. At D=256, they become almost as slow as brute force because every subtree needs to be checked.

### Annoy (Spotify)

**Annoy** (Approximate Nearest Neighbors Oh Yeah) builds a forest of random projection trees:
1. Pick two random points, draw the hyperplane between them
2. Recursively split points on either side
3. Repeat to build many trees
4. At query time, search ALL trees and merge results

Annoy trades **perfect recall for speed**. It works well for D < 1000.

| Algorithm | Time Complexity | Best For | Memory |
|-----------|----------------|----------|--------|
| Brute Force | O(N×D) | N < 1M | O(N×D) |
| KD-Tree | O(D log N) avg | D < 20 | O(N) |
| Annoy | O(D log N) avg | D < 1000 | O(N) + trees |
| FAISS IVF | O(D × N/nlist) | Any D | O(N×D) + centroids |

In [ ]:
from scipy.spatial import KDTree
import time
import numpy as np

# ======================================================================
# KD-Tree ANN Search with scipy
#
# We compare KD-tree vs brute force to show the speedup at small scale.
# Then we demonstrate why KD-trees fail at high dimensions.
# ======================================================================

def benchmark_kdtree_vs_brute(N, D):
    """Compare KD-tree and brute force search times."""
    # Generate random L2-normalized data
    data = np.random.randn(N, D).astype(np.float64)
    data = data / np.linalg.norm(data, axis=1, keepdims=True)
    
    query = np.random.randn(D).astype(np.float64)
    query = query / np.linalg.norm(query)
    
    # Build KD-tree
    t0 = time.perf_counter()
    tree = KDTree(data)
    build_time = (time.perf_counter() - t0) * 1000
    
    # KD-tree search
    n_trials = 10
    kd_times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        dists, idx = tree.query(query, k=10)
        kd_times.append((time.perf_counter() - t0) * 1000)
    kd_search_ms = np.mean(kd_times)
    
    # Brute force search
    bf_times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        sims = data @ query
        top_k = np.argsort(sims)[-10:][::-1]
        bf_times.append((time.perf_counter() - t0) * 1000)
    bf_search_ms = np.mean(bf_times)
    
    return build_time, kd_search_ms, bf_search_ms


print("=" * 75)
print("KD-Tree vs Brute Force: How Dimension Kills Tree-Based Methods")
print("=" * 75)
print(f"\n{'Dimension D':>12} {'N=10K KD-tree (ms)':>22} {'N=10K Brute (ms)':>20} {'Speedup':>10}")
print("-" * 70)

N = 10_000
for D in [8, 16, 32, 64, 128, 256]:
    bt, kd, bf = benchmark_kdtree_vs_brute(N, D)
    speedup = bf / kd if kd > 0 else float('inf')
    print(f"{D:>12} {kd:>22.3f} {bf:>20.3f} {speedup:>9.1f}x")

print()
print("Key insight: KD-tree speedup collapses as D increases.")
print("At D=256 (our embedding dim), KD-tree is NOT faster than brute force.")
print("This is the CURSE OF DIMENSIONALITY — tree partitions become useless in high-D.")
print()
print("For D=256 at 200B images, we need FAISS or ScaNN (see Section 5).")

## 3. LSH: Locality-Sensitive Hashing

### The Analogy

Imagine you want to group 200 billion photos by color. You could ask: "Is the dominant color of this photo red, blue, green, or yellow?" Each photo gets a single-letter label. Now when you have a query photo, you only search photos with the same letter — a 4× speedup.

LSH does this with random projections in high-dimensional space. Similar vectors tend to hash to the **same bucket**. You only search within the same bucket.

### How Random Projection LSH Works

1. Pick $L$ random hyperplanes (each is a random unit vector $r$)
2. For each vector $x$: compute $	ext{sign}(r \cdot x)$ — is it above or below the hyperplane?
3. Concatenate the $L$ signs into a binary hash code
4. Vectors with the same hash code land in the same bucket

**Key property:** If two vectors have high cosine similarity, they are likely to hash to the same bucket.

$$P[	ext{same hash}] = 1 - \frac{\theta}{\pi} \approx \text{cosine similarity}$$

where $\theta$ is the angle between the vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ======================================================================
# LSH: Random Projection Implementation from Scratch
#
# We implement the full LSH pipeline:
#   1. Choose L random hyperplanes
#   2. Hash all database vectors
#   3. At query time: hash the query, search only the same bucket
# ======================================================================

class RandomProjectionLSH:
    """
    Locality-Sensitive Hashing via Random Projections.
    
    Think of it as a fingerprinting system for image embeddings.
    Similar images get the same (or close) fingerprint.
    
    Instead of comparing every image to the query, we only compare
    images with matching fingerprints — a massive speedup.
    """
    
    def __init__(self, embedding_dim, n_hyperplanes=16, n_tables=4):
        """
        Args:
            embedding_dim: Dimension of input vectors
            n_hyperplanes: Number of random hyperplanes per table (= bits per hash)
            n_tables: Number of hash tables (multiple = better recall)
        """
        self.D = embedding_dim
        self.L = n_hyperplanes
        self.T = n_tables
        
        # Each table has L random hyperplanes (normal vectors)
        # Shape: (T, L, D)
        self.hyperplanes = np.random.randn(n_tables, n_hyperplanes, embedding_dim)
        # L2 normalize each hyperplane
        norms = np.linalg.norm(self.hyperplanes, axis=2, keepdims=True)
        self.hyperplanes = self.hyperplanes / norms
        
        # Hash table: dict[table_id] -> dict[hash_code -> list of indices]
        self.tables = [{} for _ in range(n_tables)]
        self.data = None
    
    def _hash(self, vectors, table_id):
        """
        Compute binary hash codes for vectors using table t.
        
        For each hyperplane: +1 if vector is on positive side, 0 otherwise.
        Combine L bits into a single integer hash code.
        """
        # Project onto each hyperplane: (N, L)
        projections = vectors @ self.hyperplanes[table_id].T
        # Binary: 1 if positive, 0 if negative
        bits = (projections > 0).astype(np.uint8)
        # Convert L bits to integer hash code
        powers = 2 ** np.arange(self.L, dtype=np.int64)
        hash_codes = bits @ powers  # (N,)
        return hash_codes
    
    def build_index(self, data):
        """Add all data vectors to the hash tables."""
        self.data = data
        N = len(data)
        
        for t in range(self.T):
            hash_codes = self._hash(data, t)
            for idx, code in enumerate(hash_codes):
                code_key = int(code)
                if code_key not in self.tables[t]:
                    self.tables[t][code_key] = []
                self.tables[t][code_key].append(idx)
        
        # Stats
        bucket_sizes = [len(v) for t_dict in self.tables for v in t_dict.values()]
        print(f"Index built: {N:,} vectors, {self.T} tables, {self.L} hyperplanes each")
        print(f"Avg bucket size: {np.mean(bucket_sizes):.1f}, Max: {max(bucket_sizes)}")
    
    def search(self, query, k=10):
        """
        Search for k nearest neighbors of query.
        
        Steps:
        1. Hash the query into each table
        2. Collect all candidates from matching buckets
        3. Compute exact similarity only for candidates
        4. Return top-k
        """
        candidates = set()
        
        for t in range(self.T):
            code = int(self._hash(query.reshape(1, -1), t)[0])
            if code in self.tables[t]:
                candidates.update(self.tables[t][code])
        
        if not candidates:
            # Fallback: if no candidates, brute force (shouldn't happen often)
            return np.argsort(self.data @ query)[-k:][::-1]
        
        candidates = list(candidates)
        candidate_vecs = self.data[candidates]
        sims = candidate_vecs @ query
        
        top_local_idx = np.argsort(sims)[-k:][::-1]
        top_global_idx = [candidates[i] for i in top_local_idx]
        top_sims = sims[top_local_idx]
        
        return np.array(top_global_idx), top_sims, len(candidates)


# ---- Demo ----
np.random.seed(42)
N, D = 50_000, 64

data = np.random.randn(N, D).astype(np.float32)
data = data / np.linalg.norm(data, axis=1, keepdims=True)

query = np.random.randn(D).astype(np.float32)
query = query / np.linalg.norm(query)

# Build LSH index
lsh = RandomProjectionLSH(embedding_dim=D, n_hyperplanes=12, n_tables=5)
lsh.build_index(data)

# Brute-force ground truth
bf_sims = data @ query
bf_top10 = set(np.argsort(bf_sims)[-10:][::-1])

# LSH search
lsh_idx, lsh_sims, n_candidates = lsh.search(query, k=10)
lsh_top10 = set(lsh_idx)

recall = len(bf_top10 & lsh_top10) / 10

print(f"\nLSH Search Results:")
print(f"  Total vectors in index: {N:,}")
print(f"  Candidates checked:     {n_candidates:,} ({100*n_candidates/N:.1f}% of index)")
print(f"  Speedup vs brute force: {N / n_candidates:.1f}x fewer distance computations")
print(f"  Recall@10:              {recall:.1%}")
print()
print("The 'approximate' part: LSH may miss some true neighbors.")
print("Increasing n_tables improves recall but reduces speedup.")

In [ ]:
# ======================================================================
# LSH Collision Probability: The Math Behind Why It Works
#
# The key theoretical property of LSH:
# P[hash(a) == hash(b)] = 1 - theta/pi
# where theta = angle between a and b
# This equals the cosine similarity when angles are small.
# ======================================================================

import numpy as np
import matplotlib.pyplot as plt

def theoretical_collision_prob(cosine_sim):
    """P[same hash] for a single random hyperplane given cosine similarity."""
    # theta = arccos(cosine_sim)
    theta = np.arccos(np.clip(cosine_sim, -1, 1))
    return 1 - theta / np.pi

def prob_collision_L_hyperplanes(cosine_sim, L):
    """P[same hash code] with L hyperplanes = all L bits match."""
    p_single = theoretical_collision_prob(cosine_sim)
    return p_single ** L

def prob_at_least_one_table(cosine_sim, L, T):
    """P[same hash in at least one of T tables]."""
    p_miss_one_table = 1 - prob_collision_L_hyperplanes(cosine_sim, L)
    return 1 - p_miss_one_table ** T

cosine_sims = np.linspace(0.0, 1.0, 200)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Effect of L (bits per table)
ax = axes[0]
for L in [4, 8, 12, 16]:
    probs = [prob_collision_L_hyperplanes(s, L) for s in cosine_sims]
    ax.plot(cosine_sims, probs, linewidth=2, label=f'L={L} bits')

ax.set_xlabel('Cosine Similarity', fontsize=13)
ax.set_ylabel('P[same hash bucket]', fontsize=13)
ax.set_title('Effect of Hash Length (L bits) on Collision Probability\n(single table)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axvline(0.8, color='gray', linestyle=':', alpha=0.7)
ax.text(0.81, 0.5, 'Typical\nsimilar pair', fontsize=9, color='gray')

# Right: Effect of T (multiple tables) for L=12
ax = axes[1]
L = 12
for T in [1, 3, 5, 10]:
    probs = [prob_at_least_one_table(s, L, T) for s in cosine_sims]
    ax.plot(cosine_sims, probs, linewidth=2, label=f'T={T} tables')

ax.set_xlabel('Cosine Similarity', fontsize=13)
ax.set_ylabel('P[retrieved] (at least 1 table matches)', fontsize=13)
ax.set_title(f'Effect of Table Count (L={L} bits fixed)\nMultiple tables = better recall', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0.95, color='#4CAF50', linestyle='--', linewidth=1.5, label='95% target')
ax.text(0.02, 0.97, '95% recall target', fontsize=9, color='#4CAF50')

plt.suptitle('LSH Collision Probabilities: More Tables = Better Recall', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key trade-off:")
print("  More hyperplanes (L): fewer false positives, higher precision, but lower recall")
print("  More tables (T):      better recall at cost of more memory and search time")
print()
print("Interview tip: Mention this recall vs. speed trade-off explicitly.")
print("In practice, FAISS is preferred over custom LSH for production systems.")

## 4. IVF: Inverted File Index (Clustering-Based ANN)

### The Analogy

Instead of searching the whole city for your dog, you first ask: "Which neighborhood does this dog usually live in?" You run to that neighborhood and only search there. This is exactly what IVF does — it clusters all embeddings into groups (using k-means) and at query time, only searches the closest clusters.

### How IVF Works

**Offline (index building):**
1. Run k-means on all embeddings to create `nlist` cluster centroids (e.g., 4096 clusters)
2. Assign each embedding to its nearest centroid
3. Store a "posting list" per centroid: which embeddings belong to this cluster?

**Online (query time):**
1. Find the `nprobe` nearest centroids to the query (e.g., top-8 of 4096)
2. Search only the embeddings in those `nprobe` clusters
3. Return the top-k most similar embeddings found

### Complexity

```
Index build time:  O(N × D × iterations)  -- k-means
Query time:        O(nlist × D + nprobe × (N/nlist) × D)
                 = O(D × (nlist + nprobe × N/nlist))
                 
With nlist=4096, nprobe=8, N=200B, D=256:
  = 256 × (4096 + 8 × 200B/4096) ≈ 256 × (4096 + 390M) ≈ manageable
```

### IVF Parameters: The Recall vs Speed Knob

| Parameter | Meaning | Effect |
|-----------|---------|--------|
| `nlist` | Number of clusters | More clusters = faster search, worse recall |
| `nprobe` | Clusters searched per query | More probes = better recall, slower search |
| Rule of thumb | `nlist = sqrt(N)` | Good starting point |

In [ ]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import time

# ======================================================================
# IVF (Inverted File Index) — Conceptual Implementation
#
# We implement IVF using sklearn k-means for the clustering step.
# In production, FAISS does this much more efficiently.
# ======================================================================

class IVFIndex:
    """
    Inverted File Index for approximate nearest neighbor search.
    
    Think of it like a phone book organized by city:
    - Instead of searching every number, you first find the city
    - Then only search numbers in that city
    
    Here 'city' = cluster centroid, 'numbers' = image embeddings.
    """
    
    def __init__(self, nlist=64):
        """
        Args:
            nlist: Number of clusters (Voronoi cells)
        """
        self.nlist = nlist
        self.kmeans = MiniBatchKMeans(n_clusters=nlist, random_state=42, n_init=3)
        self.posting_lists = {}  # centroid_id -> list of (global_idx, embedding)
        self.data = None
    
    def build(self, data):
        """
        Build the IVF index.
        
        Steps:
        1. Run k-means to find nlist cluster centroids (coarse quantizer)
        2. Assign each vector to its nearest centroid
        3. Build posting lists: centroid -> list of vectors
        """
        print(f"Building IVF index with {self.nlist} clusters...")
        print(f"  Input: {len(data):,} vectors of dimension {data.shape[1]}")
        
        t0 = time.perf_counter()
        
        # Step 1: Coarse quantization via k-means
        cluster_ids = self.kmeans.fit_predict(data)
        build_time = time.perf_counter() - t0
        
        self.data = data
        
        # Step 2: Build posting lists
        for i in range(self.nlist):
            self.posting_lists[i] = []
        
        for idx, cid in enumerate(cluster_ids):
            self.posting_lists[cid].append(idx)
        
        sizes = [len(v) for v in self.posting_lists.values()]
        print(f"  Build time: {build_time*1000:.0f}ms")
        print(f"  Avg cluster size: {np.mean(sizes):.0f}, Max: {max(sizes)}, Min: {min(sizes)}")
    
    def search(self, query, k=10, nprobe=4):
        """
        Search for k nearest neighbors using nprobe clusters.
        
        Args:
            query: (D,) query embedding
            k: Number of results to return
            nprobe: How many clusters to search (recall vs speed knob)
        """
        # Step 1: Find nprobe nearest centroids
        centroid_sims = self.kmeans.cluster_centers_ @ query
        top_centroids = np.argsort(centroid_sims)[-nprobe:][::-1]
        
        # Step 2: Search within those clusters
        candidates = []
        for cid in top_centroids:
            candidates.extend(self.posting_lists[cid])
        
        # Step 3: Exact search within candidates
        if not candidates:
            return [], [], 0
        
        candidate_vecs = self.data[candidates]
        sims = candidate_vecs @ query
        top_local = np.argsort(sims)[-k:][::-1]
        top_global = [candidates[i] for i in top_local]
        
        return np.array(top_global), sims[top_local], len(candidates)


# ---- Demo ----
np.random.seed(42)
N, D = 100_000, 64
nlist = int(np.sqrt(N))  # Rule of thumb: nlist = sqrt(N)

data = np.random.randn(N, D).astype(np.float32)
data = data / np.linalg.norm(data, axis=1, keepdims=True)

query = np.random.randn(D).astype(np.float32)
query = query / np.linalg.norm(query)

# Build IVF index
ivf = IVFIndex(nlist=nlist)
ivf.build(data)

# Ground truth (brute force)
bf_top10 = set(np.argsort(data @ query)[-10:][::-1])

# Benchmark nprobe values
print(f"\n{'nprobe':>8} {'Candidates':>12} {'Speedup':>10} {'Recall@10':>12}")
print("-" * 46)
for nprobe in [1, 2, 4, 8, 16, 32]:
    idx, sims, n_cands = ivf.search(query, k=10, nprobe=nprobe)
    recall = len(bf_top10 & set(idx)) / 10
    speedup = N / n_cands if n_cands > 0 else float('inf')
    print(f"{nprobe:>8} {n_cands:>12,} {speedup:>9.1f}x {recall:>11.1%}")

print()
print("Key insight: nprobe controls the recall vs. speed trade-off.")
print(f"nlist = sqrt(N) = {nlist} is a good rule-of-thumb starting point.")

## 5. FAISS: Industry-Standard ANN Library

### What Is FAISS?

**FAISS** (Facebook AI Similarity Search) is the gold-standard open-source library for billion-scale nearest neighbor search, built by Meta AI Research. It is used in production at Pinterest, Uber, LinkedIn, and many others.

Think of FAISS as the "NumPy of vector search" — it handles all the low-level optimizations (SIMD instructions, GPU acceleration, memory-efficient data structures) so you just call simple Python APIs.

### FAISS Index Types You Need to Know

| Index Type | Description | Best For |
|-----------|-------------|----------|
| `IndexFlatL2` | Exact brute force (L2 distance) | Baseline, small N < 1M |
| `IndexFlatIP` | Exact brute force (inner product) | Baseline with normalized vectors |
| `IndexIVFFlat` | IVF + exact search per cluster | Good accuracy, large scale |
| `IndexIVFPQ` | IVF + product quantization | Memory-efficient, 200B scale |
| `IndexHNSW` | Hierarchical Navigable Small World | Very fast queries, moderate memory |

### FAISS IVFFlat: The Workhorse Index

For our 200B image system:
```
nlist  = 65,536  (sqrt of ~4 billion, we shard the full 200B)
nprobe = 128     (search 0.2% of clusters per query)
Expected recall: ~95%
Expected speed:  ~10ms per query on CPU
```

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

# ======================================================================
# FAISS Demo: Build IVFFlat Index, Search, Measure Recall vs Speed
#
# We try to import faiss; if unavailable we simulate results using
# our IVFIndex from Section 4. The API signatures are identical.
# Install: pip install faiss-cpu
# ======================================================================

try:
    import faiss
    FAISS_AVAILABLE = True
    print("FAISS is available — using real FAISS index.")
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS not installed (pip install faiss-cpu to get it).")
    print("Simulating FAISS behavior with our IVFIndex implementation.")
    print()


def build_faiss_ivfflat(data, nlist):
    """Build a FAISS IVFFlat index (or simulate it)."""
    N, D = data.shape
    if FAISS_AVAILABLE:
        # FAISS: normalize data for cosine similarity (use IP index)
        faiss.normalize_L2(data)  # in-place L2 normalization
        
        # Coarse quantizer (flat index for centroids)
        quantizer = faiss.IndexFlatIP(D)
        
        # IVF index: inner product similarity after normalization = cosine sim
        index = faiss.IndexIVFFlat(quantizer, D, nlist, faiss.METRIC_INNER_PRODUCT)
        
        # Train (run k-means to find centroids)
        index.train(data)
        
        # Add all vectors
        index.add(data)
        
        return index
    else:
        # Simulation with our IVFIndex
        idx = IVFIndex(nlist=nlist)
        idx.build(data)
        return idx


def search_index(index, query, k=10, nprobe=8):
    """Search the index (FAISS or simulated)."""
    if FAISS_AVAILABLE:
        index.nprobe = nprobe
        q = query.reshape(1, -1).copy()
        faiss.normalize_L2(q)
        distances, indices = index.search(q, k)
        return indices[0], distances[0]
    else:
        idx_arr, sims, n_cands = index.search(query, k=k, nprobe=nprobe)
        return idx_arr, sims


# ---- Build index ----
np.random.seed(42)
N, D = 200_000, 256
nlist = 512  # ~sqrt(200k), a good starting point

print(f"Building index: N={N:,}, D={D}, nlist={nlist}")
data = np.random.randn(N, D).astype(np.float32)
data = data / np.linalg.norm(data, axis=1, keepdims=True)

t0 = time.perf_counter()
index = build_faiss_ivfflat(data.copy(), nlist)
build_ms = (time.perf_counter() - t0) * 1000
print(f"Index built in {build_ms:.0f}ms")

# Ground truth
query = np.random.randn(D).astype(np.float32)
query = query / np.linalg.norm(query)
bf_top10 = set(np.argsort(data @ query)[-10:][::-1])

# ---- Recall vs Speed benchmark ----
nprobe_values = [1, 2, 4, 8, 16, 32, 64, 128]
recalls = []
latencies_ms = []

print(f"\n{'nprobe':>8} {'Latency (ms)':>14} {'Recall@10':>12}")
print("-" * 38)

for nprobe in nprobe_values:
    # Measure latency over multiple trials
    trial_times = []
    for _ in range(20):
        t0 = time.perf_counter()
        result_idx, result_sims = search_index(index, query, k=10, nprobe=nprobe)
        trial_times.append((time.perf_counter() - t0) * 1000)
    
    avg_ms = np.mean(trial_times)
    recall = len(bf_top10 & set(result_idx)) / 10
    recalls.append(recall)
    latencies_ms.append(avg_ms)
    print(f"{nprobe:>8} {avg_ms:>14.3f} {recall:>11.1%}")

# ---- Plot Recall vs Latency ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Recall vs nprobe
ax = axes[0]
ax.plot(nprobe_values, recalls, 'o-', color='#1E88E5', linewidth=2.5, markersize=8)
ax.set_xlabel('nprobe', fontsize=13)
ax.set_ylabel('Recall@10', fontsize=13)
ax.set_title('Recall@10 vs nprobe\n(more probes = better recall)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
ax.axhline(0.95, color='#4CAF50', linestyle='--', label='95% recall target')
ax.legend(fontsize=11)

# Right: Recall vs Latency (the key trade-off curve)
ax = axes[1]
ax.plot(latencies_ms, recalls, 's-', color='#E53935', linewidth=2.5, markersize=8)
for i, (lat, rec, nprb) in enumerate(zip(latencies_ms, recalls, nprobe_values)):
    if nprb in [1, 8, 32, 128]:
        ax.annotate(f'nprobe={nprb}', (lat, rec), textcoords='offset points',
                    xytext=(5, 5), fontsize=9)
ax.set_xlabel('Query Latency (ms)', fontsize=13)
ax.set_ylabel('Recall@10', fontsize=13)
ax.set_title('Recall-Latency Trade-off Curve\n(the key ANN engineering decision)', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.suptitle('FAISS IVFFlat: Recall vs Speed Trade-off', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterview talking points:")
print("  - nprobe is the main knob: higher nprobe = better recall, higher latency")
print("  - Target 95% Recall@10 with < 50ms latency for production systems")
print("  - FAISS supports GPU acceleration for 10-100x additional speedup")
print("  - ScaNN (Google) uses similar ideas with additional optimizations")

## 6. Product Quantization: Memory Optimization

### The Problem: Memory at Scale

200 billion vectors × 256 dimensions × 4 bytes (float32) = **200 TB of RAM**.

That is not feasible. We need to compress embeddings dramatically while preserving search quality.

### The Analogy

Imagine storing addresses. Instead of writing "1234 Oak Street, San Francisco, CA 94102", you use a code: "SF-SOMA-BLOCK-42". Product quantization does exactly this — it encodes each vector as a short sequence of codebook indices.

### How Product Quantization (PQ) Works

1. **Split** the D-dimensional vector into M sub-vectors of D/M dimensions each
2. **Cluster** each sub-space independently with k-means (creating M codebooks)
3. **Encode** each sub-vector as the index of its nearest centroid (e.g., 8 bits = 256 centroids per sub-space)
4. **Store** only the M indices instead of D float32 values

```
Original vector:  256 dims × 4 bytes = 1024 bytes
PQ compressed:    M=32 sub-vectors × 8 bits = 32 bytes
Compression ratio: 32× \!

For 200B images:
  Full float32: 200B × 1024 bytes = 200 TB
  With PQ:      200B × 32 bytes  = 6.4 TB  (fits in ~1,000 machines with 8GB RAM each)
```

### The Trade-off

PQ introduces **quantization error** — the reconstructed vector is not exactly the original. This slightly reduces search accuracy (recall), but the memory savings make it essential at scale.

| Variant | Sub-vectors (M) | Bits/sub | Memory per vector | Recall |
|---------|----------------|----------|-------------------|--------|
| PQ32x8 | 32 | 8 | 32 bytes | ~90% |
| PQ64x8 | 64 | 8 | 64 bytes | ~95% |
| SQ8 (Scalar) | 1 | 8/dim | 64 bytes | ~99% |
| Float16 | — | 16/dim | 128 bytes | ~99.9% |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans

# ======================================================================
# Product Quantization: Conceptual Implementation
#
# We implement PQ from scratch to show the encode/decode mechanism.
# In production, FAISS's IndexIVFPQ does this with heavy optimization.
# ======================================================================

class ProductQuantizer:
    """
    Product Quantization for memory-efficient vector storage.
    
    Splits each vector into M sub-vectors and encodes each sub-vector
    as an index into a learned codebook of k* centroids.
    
    Memory: M * log2(k*) bits per vector instead of D * 32 bits.
    """
    
    def __init__(self, D, M=8, k_star=256):
        """
        Args:
            D: Original embedding dimension
            M: Number of sub-quantizers (must divide D evenly)
            k_star: Number of centroids per sub-quantizer (256 = 8 bits)
        """
        assert D % M == 0, f"D ({D}) must be divisible by M ({M})"
        self.D = D
        self.M = M
        self.k_star = k_star
        self.sub_dim = D // M
        self.codebooks = None  # Shape: (M, k_star, sub_dim)
    
    def train(self, data):
        """Learn M codebooks via k-means on each sub-space."""
        N = len(data)
        self.codebooks = np.zeros((self.M, self.k_star, self.sub_dim), dtype=np.float32)
        
        for m in range(self.M):
            # Extract the m-th sub-vector from each data point
            sub_vectors = data[:, m * self.sub_dim : (m+1) * self.sub_dim]
            
            # K-means to find k* centroids for this sub-space
            km = MiniBatchKMeans(n_clusters=self.k_star, random_state=42, n_init=1, max_iter=100)
            km.fit(sub_vectors)
            self.codebooks[m] = km.cluster_centers_.astype(np.float32)
    
    def encode(self, data):
        """
        Encode vectors into PQ codes.
        Returns: (N, M) array of uint8 centroid indices.
        """
        N = len(data)
        codes = np.zeros((N, self.M), dtype=np.uint8)
        
        for m in range(self.M):
            sub_vectors = data[:, m * self.sub_dim : (m+1) * self.sub_dim]  # (N, sub_dim)
            centroids = self.codebooks[m]  # (k_star, sub_dim)
            
            # Assign each sub-vector to nearest centroid
            # Use negative dot product as approximate distance (for normalized vectors)
            sims = sub_vectors @ centroids.T  # (N, k_star)
            codes[:, m] = np.argmax(sims, axis=1).astype(np.uint8)
        
        return codes
    
    def decode(self, codes):
        """
        Reconstruct approximate vectors from PQ codes.
        Returns: (N, D) array of approximate float32 vectors.
        """
        N = len(codes)
        reconstructed = np.zeros((N, self.D), dtype=np.float32)
        
        for m in range(self.M):
            centroid_idx = codes[:, m].astype(int)
            reconstructed[:, m * self.sub_dim : (m+1) * self.sub_dim] = self.codebooks[m][centroid_idx]
        
        return reconstructed
    
    def memory_usage(self, N):
        """Compute memory usage for N vectors."""
        original_bytes = N * self.D * 4  # float32
        pq_bytes = N * self.M * 1       # 1 byte per code (uint8)
        codebook_bytes = self.M * self.k_star * self.sub_dim * 4  # float32
        return original_bytes, pq_bytes, codebook_bytes


# ---- Demo ----
np.random.seed(42)
N_train, N_test, D = 50_000, 1_000, 64
M = 8
k_star = 256

print(f"Product Quantization Demo")
print(f"  D={D}, M={M} sub-quantizers, k*={k_star} centroids each")
print()

# Generate data
train_data = np.random.randn(N_train, D).astype(np.float32)
train_data = train_data / np.linalg.norm(train_data, axis=1, keepdims=True)
test_data = np.random.randn(N_test, D).astype(np.float32)
test_data = test_data / np.linalg.norm(test_data, axis=1, keepdims=True)

# Train PQ
print("Training PQ codebooks...")
pq = ProductQuantizer(D=D, M=M, k_star=k_star)
pq.train(train_data)

# Encode and decode test data
codes = pq.encode(test_data)
reconstructed = pq.decode(codes)

# Compute reconstruction error
errors = np.linalg.norm(test_data - reconstructed, axis=1)
cosine_sim_before = np.sum(test_data * test_data, axis=1)  # always 1 (normalized)
cosine_sim_after = np.sum(test_data * reconstructed, axis=1)

print(f"Reconstruction Quality:")
print(f"  Avg L2 reconstruction error: {errors.mean():.4f}")
print(f"  Avg cosine sim (orig vs reconstructed): {cosine_sim_after.mean():.4f}")
print()

# Memory analysis
orig_bytes, pq_bytes, cb_bytes = pq.memory_usage(200_000_000_000)
print(f"Memory at 200B images scale:")
print(f"  Original float32:      {orig_bytes / 1e12:.1f} TB")
print(f"  PQ codes only:         {pq_bytes / 1e12:.1f} TB")
print(f"  Codebooks (overhead):  {cb_bytes / 1e6:.1f} MB  (negligible)")
print(f"  Compression ratio:     {orig_bytes / pq_bytes:.0f}x")
print()
print(f"Bits per vector: M * log2(k*) = {M} * {int(np.log2(k_star))} = {M * int(np.log2(k_star))} bits")
print(f"vs. original:    D * 32 = {D * 32} bits")
print(f"Compression:     {D * 32 / (M * int(np.log2(k_star))):.0f}x reduction")

## 7. Complete Prediction Pipeline

### The Full Flow (Real-Time, < 100ms)

```
User uploads query image
        |
        v
[1] Preprocessing Service
    - Resize to 256x256, center crop 224x224
    - Normalize (ImageNet mean/std)
    - Ensure RGB
        |
        v
[2] Embedding Generation Service  (~20ms)
    - Run preprocessed image through CNN/ViT
    - Apply projection head
    - L2 normalize -> 256-D embedding vector
        |
        v
[3] ANN Search Service  (~10ms)
    - Query the FAISS index
    - Parameters: nprobe=128, k=500 (retrieve more than needed for re-ranking)
    - Returns 500 candidate (image_id, similarity_score) pairs
        |
        v
[4] Re-ranking Service  (~5ms)
    - Remove duplicates and near-duplicates
    - Filter private/deleted images
    - Filter policy-violating content
    - Apply diversity constraints (don't show 10 identical-looking results)
    - Return top-K final results (e.g., K=50)
        |
        v
User sees ranked results
```

Total budget: ~35ms compute + network overhead = well under 100ms.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import time

# ======================================================================
# Complete Prediction Pipeline Simulation
#
# We simulate each stage with realistic timing targets.
# In production each stage would be a separate microservice.
# ======================================================================

# ---- Stage 1: Preprocessing ----
preprocess_pipeline = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def preprocess_image(pil_image):
    """Stage 1: Preprocess a PIL image into a tensor."""
    rgb_image = pil_image.convert('RGB')
    tensor = preprocess_pipeline(rgb_image)  # (3, 224, 224)
    return tensor.unsqueeze(0)  # (1, 3, 224, 224) — add batch dimension


# ---- Stage 2: Embedding Model (simplified for demo) ----
class EmbeddingService(nn.Module):
    """
    Lightweight embedding model for the demo.
    In production this is ResNet-50 or ViT-B/16.
    """
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(64 * 16, 512), nn.ReLU(),
            nn.Linear(512, embedding_dim),
        )
    
    def forward(self, x):
        emb = self.net(x)
        return F.normalize(emb, p=2, dim=1)


# ---- Stage 3: ANN Search (simulated) ----
class ANNSearchService:
    """
    Simulates a FAISS IVFFlat index.
    Returns candidate (image_id, similarity) pairs.
    """
    def __init__(self, index_embeddings, image_ids):
        self.index = index_embeddings  # (N, D)
        self.image_ids = image_ids
    
    def search(self, query_embedding, k=500):
        """Return top-k candidates with their similarity scores."""
        sims = self.index @ query_embedding.flatten()
        top_k_local = np.argsort(sims)[-k:][::-1]
        return [(self.image_ids[i], float(sims[i])) for i in top_k_local]


# ---- Stage 4: Re-ranking Service ----
class ReRankingService:
    """
    Applies business logic to filter and re-rank candidates.
    
    Rules (in priority order):
    1. Remove deleted or private images
    2. Remove near-duplicates (similarity > 0.99 to already selected)
    3. Apply content policy (simulated)
    4. Apply diversity: don't select too many from same owner
    5. Return top-K
    """
    
    def __init__(self, private_image_ids=None, deleted_image_ids=None):
        self.private_ids = set(private_image_ids or [])
        self.deleted_ids = set(deleted_image_ids or [])
    
    def rerank(self, candidates, query_embedding, index_embeddings, image_id_to_idx,
               k=50, dedup_threshold=0.98, max_per_owner=3):
        """
        candidates: list of (image_id, similarity_score)
        Returns filtered, deduplicated, diverse top-k results.
        """
        selected = []
        owner_counts = {}
        selected_embeddings = []
        
        for img_id, sim_score in candidates:
            # Filter 1: deleted images
            if img_id in self.deleted_ids:
                continue
            
            # Filter 2: private images
            if img_id in self.private_ids:
                continue
            
            # Filter 3: near-duplicate removal
            if selected_embeddings:
                idx = image_id_to_idx[img_id]
                img_emb = index_embeddings[idx]
                max_sim_to_selected = max(
                    float(img_emb @ sel_emb) for sel_emb in selected_embeddings
                )
                if max_sim_to_selected > dedup_threshold:
                    continue  # Too similar to already-selected image
            
            # Filter 4: owner diversity
            owner_id = img_id // 1000  # Simulated: owner = img_id // 1000
            if owner_counts.get(owner_id, 0) >= max_per_owner:
                continue
            
            # Accept this candidate
            idx = image_id_to_idx[img_id]
            selected_embeddings.append(index_embeddings[idx])
            owner_counts[owner_id] = owner_counts.get(owner_id, 0) + 1
            selected.append({'image_id': img_id, 'similarity': sim_score})
            
            if len(selected) >= k:
                break
        
        return selected


# ============================================================
# RUN THE FULL PIPELINE
# ============================================================
np.random.seed(42)
torch.manual_seed(42)

N_index = 10_000
D = 256

# Simulate an image index
index_embeddings = np.random.randn(N_index, D).astype(np.float32)
index_embeddings = index_embeddings / np.linalg.norm(index_embeddings, axis=1, keepdims=True)
image_ids = list(range(1, N_index + 1))
image_id_to_idx = {img_id: i for i, img_id in enumerate(image_ids)}

# Simulate some private and deleted images
private_ids = set(np.random.choice(image_ids, 200, replace=False))
deleted_ids = set(np.random.choice(image_ids, 100, replace=False))

# Initialize services
embed_model = EmbeddingService(embedding_dim=D)
embed_model.eval()

ann_service = ANNSearchService(index_embeddings, image_ids)
rerank_service = ReRankingService(private_ids, deleted_ids)

# Simulate a query image
dummy_img = Image.fromarray(np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8))

print("=" * 60)
print("PREDICTION PIPELINE — End to End Trace")
print("=" * 60)

pipeline_start = time.perf_counter()

# Stage 1: Preprocessing
t0 = time.perf_counter()
tensor = preprocess_image(dummy_img)
stage1_ms = (time.perf_counter() - t0) * 1000
print(f"\n[Stage 1] Preprocessing: {stage1_ms:.2f}ms")
print(f"          Input: PIL Image (300x400x3)")
print(f"          Output: Tensor {tuple(tensor.shape)}")

# Stage 2: Embedding
t0 = time.perf_counter()
with torch.no_grad():
    query_emb = embed_model(tensor).numpy().flatten()
stage2_ms = (time.perf_counter() - t0) * 1000
print(f"\n[Stage 2] Embedding generation: {stage2_ms:.2f}ms")
print(f"          Output: {D}-D unit vector, norm={np.linalg.norm(query_emb):.4f}")

# Stage 3: ANN Search
t0 = time.perf_counter()
candidates = ann_service.search(query_emb, k=500)
stage3_ms = (time.perf_counter() - t0) * 1000
print(f"\n[Stage 3] ANN search: {stage3_ms:.2f}ms")
print(f"          Retrieved {len(candidates)} candidates")
print(f"          Top similarity: {candidates[0][1]:.4f}, Bottom: {candidates[-1][1]:.4f}")

# Stage 4: Re-ranking
t0 = time.perf_counter()
final_results = rerank_service.rerank(
    candidates, query_emb, index_embeddings, image_id_to_idx, k=50
)
stage4_ms = (time.perf_counter() - t0) * 1000

total_ms = (time.perf_counter() - pipeline_start) * 1000

print(f"\n[Stage 4] Re-ranking: {stage4_ms:.2f}ms")
print(f"          Input candidates: {len(candidates)}")
print(f"          After privacy filter: removed {len(private_ids)} private images")
print(f"          After delete filter: removed {len(deleted_ids)} deleted images")
print(f"          After dedup: near-duplicate images removed")
print(f"          Final results: {len(final_results)} images")

print(f"\n{'='*60}")
print(f"TOTAL PIPELINE TIME: {total_ms:.2f}ms")
print(f"  Stage 1 (preprocess):   {stage1_ms:6.2f}ms")
print(f"  Stage 2 (embed):        {stage2_ms:6.2f}ms")
print(f"  Stage 3 (ANN search):   {stage3_ms:6.2f}ms")
print(f"  Stage 4 (re-rank):      {stage4_ms:6.2f}ms")
print(f"{'='*60}")
print(f"\nProduction note: Stage 2 runs on GPU (~5-15ms for ResNet-50).")
print(f"Network latency adds ~10-30ms. Total budget: < 100ms.")

## 8. Indexing Pipeline

### What Is the Indexing Pipeline?

The indexing pipeline runs **offline** (continuously in the background). Its job is to:

1. Take every image on the platform (200 billion)
2. Generate embeddings in batch
3. Build the FAISS index from those embeddings
4. When new images are uploaded, add them to the index

### Two Sub-Pipelines

**Batch Indexing (initial build and periodic rebuild):**
```
All platform images
    -> Distributed preprocessing (resize, normalize)
    -> Batch embedding generation (GPU fleet)
    -> Distributed k-means for IVF centroids
    -> Build FAISS index shards
    -> Store shards in distributed storage (S3/HDFS)
    -> Load shards into ANN search servers
```

**Incremental Indexing (new image uploads):**
```
New image uploaded by user
    -> Add to preprocessing queue
    -> Embedding generated (within minutes)
    -> Assign to existing IVF cluster (no full rebuild)
    -> Add to posting list for that cluster
    -> Image becomes searchable
```

### Key Engineering Decisions

| Decision | Choice | Reason |
|----------|--------|--------|
| Batch embed frequency | Nightly rebuild | Model changes slowly |
| Incremental update | Real-time (minutes) | New images should be discoverable |
| Index sharding | By image ID range | Even distribution |
| Replication | 3x replication | Fault tolerance |
| Consistency | Eventually consistent | 200B images can't be perfectly synced |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================================================================
# Architecture Diagram: Indexing Pipeline
# ======================================================================

fig, axes = plt.subplots(2, 1, figsize=(18, 14))

# ---- TOP: Batch Indexing Pipeline ----
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.set_title('Batch Indexing Pipeline (Nightly Rebuild)', fontsize=15, fontweight='bold', pad=12)
ax.axis('off')

batch_boxes = [
    (0.2, 1.8, 'Image\nStorage\n(S3/HDFS)', '#FFE0B2', 1.4),
    (1.9, 1.8, 'Distributed\nPreprocessing\n(Spark)', '#B3E5FC', 1.7),
    (3.9, 1.8, 'GPU Fleet\nEmbedding\nGeneration', '#C8E6C9', 1.7),
    (5.9, 1.8, 'Distributed\nk-means\n(IVF Training)', '#F8BBD0', 1.7),
    (7.9, 1.8, 'FAISS\nIndex\nBuilder', '#D1C4E9', 1.5),
    (9.7, 1.8, 'Index\nShards\n(S3)', '#FFF9C4', 1.5),
]

prev_x_end = None
for (x, y, text, color, width) in batch_boxes:
    box = FancyBboxPatch((x, y), width, 1.4, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#444', linewidth=2)
    ax.add_patch(box)
    ax.text(x + width/2, y + 0.7, text, ha='center', va='center',
            fontsize=9, fontweight='bold', multialignment='center')
    if prev_x_end is not None:
        ax.annotate('', xy=(x, y+0.7), xytext=(prev_x_end, y+0.7),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))
    prev_x_end = x + width

# Stats annotations below boxes
stats = [
    (0.9, 1.4, '200B images'),
    (2.75, 1.4, '10K CPU cores'),
    (4.75, 1.4, '1000 A100 GPUs'),
    (6.75, 1.4, 'nlist=65536'),
    (8.65, 1.4, 'Sharded'),
]
for (x, y, text) in stats:
    ax.text(x, y, text, ha='center', fontsize=8, color='#555', style='italic')

# ANN Server loading
ann_box = FancyBboxPatch((9.7, 0.1), 1.5, 0.9, boxstyle='round,pad=0.08',
                         facecolor='#FFCCBC', edgecolor='#444', linewidth=2)
ax.add_patch(ann_box)
ax.text(10.45, 0.55, 'ANN Search\nServers', ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(10.45, 1.0), xytext=(10.45, 1.8),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2, linestyle='dashed'))
ax.text(10.45, 1.55, 'Load', ha='center', fontsize=8, color='#E65100')

ax.text(6.0, 4.3, 'Timeline: 200B embeddings × 0.01s/embed / 1000 GPUs ≈ 2,000s ≈ 33min (full rebuild)',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='#E8EAF6', alpha=0.9))

# ---- BOTTOM: Incremental (Real-time) Indexing ----
ax2 = axes[1]
ax2.set_xlim(0, 12)
ax2.set_ylim(0, 5)
ax2.set_title('Incremental Indexing Pipeline (New Image Uploads, Minutes Latency)',
              fontsize=15, fontweight='bold', pad=12)
ax2.axis('off')

inc_boxes = [
    (0.2, 1.8, 'User Uploads\nNew Image', '#FFCCBC', 1.6),
    (2.1, 1.8, 'Image\nQueue\n(Kafka)', '#B3E5FC', 1.5),
    (3.9, 1.8, 'Embedding\nWorker\n(GPU)', '#C8E6C9', 1.6),
    (5.8, 1.8, 'Assign to\nNearest\nIVF Cluster', '#F8BBD0', 1.7),
    (7.8, 1.8, 'Update\nPosting\nList', '#D1C4E9', 1.5),
    (9.6, 1.8, 'Image is\nSearchable\!', '#E8F5E9', 1.5),
]

prev_x_end = None
for (x, y, text, color, width) in inc_boxes:
    box = FancyBboxPatch((x, y), width, 1.4, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#444', linewidth=2)
    ax2.add_patch(box)
    ax2.text(x + width/2, y + 0.7, text, ha='center', va='center',
             fontsize=9, fontweight='bold', multialignment='center')
    if prev_x_end is not None:
        ax2.annotate('', xy=(x, y+0.7), xytext=(prev_x_end, y+0.7),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2))
    prev_x_end = x + width

timings = [
    (1.0, 1.4, 't=0s'),
    (2.85, 1.4, 't~1s'),
    (4.7, 1.4, 't~10s'),
    (6.65, 1.4, 't~15s'),
    (8.55, 1.4, 't~30s'),
    (10.35, 1.4, 't<5min'),
]
for (x, y, text) in timings:
    ax2.text(x, y, text, ha='center', fontsize=8, color='#555', style='italic')

ax2.text(6.0, 4.3, 'Key insight: No full index rebuild needed — just add to the existing cluster posting list.',
         ha='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round', facecolor='#F3E5F5', alpha=0.9))

plt.tight_layout()
plt.savefig('indexing_pipeline.png', dpi=130, bbox_inches='tight')
plt.show()

# ---- Batch Embedding Throughput Analysis ----
print("Batch Embedding Throughput Analysis")
print("=" * 55)
print()

gpu_throughput_imgs_per_sec = 500  # images/sec per A100 GPU (realistic for ResNet-50)
total_images = 200_000_000_000
n_gpus_list = [100, 500, 1000, 2000, 5000]

print(f"{'GPUs':>8} {'Time (hrs)':>12} {'Time (min)':>12} {'Feasible?':>12}")
print("-" * 48)
for n_gpus in n_gpus_list:
    total_throughput = n_gpus * gpu_throughput_imgs_per_sec
    seconds = total_images / total_throughput
    hours = seconds / 3600
    minutes = seconds / 60
    feasible = 'YES' if hours < 24 else 'NO'
    print(f"{n_gpus:>8,} {hours:>12.1f} {minutes:>12.0f} {feasible:>12}")

print()
print(f"Assumption: {gpu_throughput_imgs_per_sec} images/sec per A100 GPU (ResNet-50 batch=128)")
print("In practice Pinterest uses ~1,000 GPUs for their embedding pipeline.")

## 9. Re-Ranking Service: Business Logic

### Why Re-Rank After ANN Search?

The ANN index only knows about **visual similarity** — it has no awareness of business rules. The re-ranking service is the "bouncer" that applies real-world constraints:

1. **Duplicate removal:** If two images look nearly identical (similarity > 0.99), showing both is redundant
2. **Private image filtering:** Users can set images to private — they must never appear in others' search results
3. **Deleted image filtering:** Images may be deleted after the index was built (eventually consistent)
4. **Content policy:** Block images flagged by the content moderation system
5. **Diversity enforcement:** Avoid showing 10 near-identical results (e.g., 10 nearly identical red dresses from one store)

### Staff-Level Detail: Positional Bias

A senior candidate will mention that the re-ranking stage should also handle **positional bias** — users click items at the top of the results list more often, regardless of true relevance. Solutions:

- **Inverse Propensity Scoring (IPS):** Weight clicks by `1 / P(click given position)` to debias training data
- **Randomized experiments:** Occasionally shuffle results to collect unbiased click data

In [ ]:
import numpy as np
import pandas as pd

# ======================================================================
# Re-Ranking Service: Full Business Logic Simulation
#
# Simulates the real-world filtering steps that happen after ANN search.
# ======================================================================

np.random.seed(42)

N_candidates = 200  # ANN returned 200 candidates

# Simulate candidates with metadata
image_ids = np.arange(1, N_candidates + 1)
similarity_scores = np.sort(np.random.uniform(0.5, 0.99, N_candidates))[::-1]
owner_ids = np.random.randint(1, 50, N_candidates)  # 50 different owners

# Simulate near-duplicates (groups of 3-4 very similar images)
for i in range(0, 30, 4):
    similarity_scores[i+1:i+4] = similarity_scores[i] - np.random.uniform(0.001, 0.005, 3)

# Business rule datasets
private_images = set(image_ids[np.random.choice(N_candidates, 20, replace=False)])
deleted_images = set(image_ids[np.random.choice(N_candidates, 10, replace=False)])
policy_flagged = set(image_ids[np.random.choice(N_candidates, 8, replace=False)])

# Simulated embeddings for dedup check
embeddings = np.random.randn(N_candidates, 64).astype(np.float32)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
# Make some near-duplicates
for i in range(0, 30, 4):
    for j in range(1, 4):
        if i+j < N_candidates:
            embeddings[i+j] = embeddings[i] + np.random.randn(64) * 0.02
            embeddings[i+j] /= np.linalg.norm(embeddings[i+j])

print("Re-Ranking Service: Filtering Steps")
print("=" * 55)
print(f"Input: {N_candidates} ANN candidates")
print()

remaining = list(range(N_candidates))

# Step 1: Filter deleted images
before = len(remaining)
remaining = [i for i in remaining if image_ids[i] not in deleted_images]
print(f"Step 1: Delete filter:     {before:>4} -> {len(remaining):>4}  (-{before-len(remaining)} deleted)")

# Step 2: Filter private images
before = len(remaining)
remaining = [i for i in remaining if image_ids[i] not in private_images]
print(f"Step 2: Privacy filter:    {before:>4} -> {len(remaining):>4}  (-{before-len(remaining)} private)")

# Step 3: Filter policy-violating content
before = len(remaining)
remaining = [i for i in remaining if image_ids[i] not in policy_flagged]
print(f"Step 3: Policy filter:     {before:>4} -> {len(remaining):>4}  (-{before-len(remaining)} policy)")

# Step 4: Near-duplicate removal (greedy, using cosine similarity)
before = len(remaining)
selected = []
selected_embs = []
DEDUP_THRESHOLD = 0.97

for idx in remaining:
    if selected_embs:
        max_sim = max(float(embeddings[idx] @ sel_emb) for sel_emb in selected_embs)
        if max_sim > DEDUP_THRESHOLD:
            continue
    selected.append(idx)
    selected_embs.append(embeddings[idx])

remaining = selected
print(f"Step 4: Dedup (sim>{DEDUP_THRESHOLD}): {before:>4} -> {len(remaining):>4}  (-{before-len(remaining)} near-dupes)")

# Step 5: Owner diversity (max 3 per owner)
before = len(remaining)
owner_count = {}
diverse = []
MAX_PER_OWNER = 3

for idx in remaining:
    oid = owner_ids[idx]
    if owner_count.get(oid, 0) < MAX_PER_OWNER:
        diverse.append(idx)
        owner_count[oid] = owner_count.get(oid, 0) + 1

remaining = diverse
print(f"Step 5: Diversity (max {MAX_PER_OWNER}/owner): {before:>4} -> {len(remaining):>4}  (-{before-len(remaining)} diversity cut)")

# Final top-50
final_50 = remaining[:50]
print(f"\nFinal output: top {len(final_50)} results")
print()

# Show final results table
final_df = pd.DataFrame({
    'rank': range(1, len(final_50)+1),
    'image_id': image_ids[final_50],
    'similarity': similarity_scores[final_50].round(4),
    'owner_id': owner_ids[final_50],
})
print("Top 15 final results:")
print(final_df.head(15).to_string(index=False))

print()
print("Pipeline summary:")
print(f"  Input candidates:  {N_candidates}")
print(f"  Output results:    {len(final_50)}")
print(f"  Filter ratio:      {100*(N_candidates-len(final_50))/N_candidates:.0f}% of candidates removed")

## 10. Scale Analysis: Infrastructure for 200B Images

### Back-of-Envelope Calculations (Do These in Interviews\!)

These calculations show you understand the engineering constraints at scale. Always do them in your interview — it separates staff from senior candidates.

### Storage for Embeddings

```
200B images × 256 dims × 4 bytes (float32) = 204.8 TB

With Product Quantization (PQ32x8):
200B × 32 bytes = 6.4 TB

Distributed across servers with 512 GB RAM each:
  Float32: 204.8 TB / 512 GB = ~400 servers
  PQ:      6.4 TB / 512 GB  = ~13 servers
```

### Query Throughput Requirements

```
Pinterest scale: ~1B queries/day
= 1B / 86,400 = ~11,600 QPS (queries per second)
Peak factor 5x = ~58,000 QPS

Each ANN server handles ~2,000-5,000 QPS
=> Need 12 - 29 ANN search servers (at PQ scale)
```

### Embedding Generation (Inference)

```
New images uploaded: ~1M/day = ~12 new images/second
ResNet-50 on A100: ~500 images/sec per GPU
=> Need 12/500 = 0.024 GPUs continuously = 1 GPU for incremental updates

For nightly full rebuild of 200B images:
=> Need ~1,000 GPUs for a 30-minute rebuild window
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ======================================================================
# Scale Analysis: Infrastructure Calculator
#
# All the numbers you need to quote in a system design interview.
# ======================================================================

print("=" * 70)
print("VISUAL SEARCH SYSTEM: 200B IMAGE SCALE ANALYSIS")
print("=" * 70)

# --- Constants ---
N_IMAGES = 200_000_000_000   # 200B images
EMB_DIM = 256
FLOAT32_BYTES = 4
PQ_BYTES_PER_VECTOR = 32     # PQ32x8 encoding
QPS_DAILY = 1_000_000_000 / 86400  # 1B queries/day in QPS
PEAK_FACTOR = 5
GPU_INFERENCE_RATE = 500     # images/sec per A100 (ResNet-50, batch=128)
NEW_IMAGES_PER_DAY = 1_000_000

print("\n1. EMBEDDING STORAGE")
print("-" * 45)
raw_tb = N_IMAGES * EMB_DIM * FLOAT32_BYTES / 1e12
pq_tb = N_IMAGES * PQ_BYTES_PER_VECTOR / 1e12
fp16_tb = N_IMAGES * EMB_DIM * 2 / 1e12

print(f"  Float32 (raw):     {raw_tb:>8.1f} TB")
print(f"  Float16:           {fp16_tb:>8.1f} TB")
print(f"  PQ32x8 (8 bits):   {pq_tb:>8.1f} TB")
print(f"  PQ compression:    {raw_tb/pq_tb:>7.0f}x vs float32")

ram_per_server_gb = 512
n_servers_raw = np.ceil(raw_tb * 1000 / ram_per_server_gb)
n_servers_pq = np.ceil(pq_tb * 1000 / ram_per_server_gb)
print(f"\n  Servers needed ({ram_per_server_gb}GB RAM each):")
print(f"    Float32:  {n_servers_raw:.0f} servers")
print(f"    PQ32x8:   {n_servers_pq:.0f} servers  << 30x fewer machines\!")

print("\n2. QUERY SERVING REQUIREMENTS")
print("-" * 45)
avg_qps = QPS_DAILY
peak_qps = avg_qps * PEAK_FACTOR
qps_per_server = 3000  # Realistic ANN server throughput
n_ann_servers = np.ceil(peak_qps / qps_per_server)

print(f"  Average QPS:       {avg_qps:>10.0f}")
print(f"  Peak QPS (5x):     {peak_qps:>10.0f}")
print(f"  QPS per server:    {qps_per_server:>10,}")
print(f"  ANN servers needed:{n_ann_servers:>10.0f}  (with 3x replication: {n_ann_servers*3:.0f})")

print("\n3. EMBEDDING GENERATION (INFERENCE)")
print("-" * 45)
new_img_per_sec = NEW_IMAGES_PER_DAY / 86400
gpus_incremental = np.ceil(new_img_per_sec / GPU_INFERENCE_RATE)
rebuild_window_min = 30
total_rebuild_time_s = N_IMAGES / GPU_INFERENCE_RATE
gpus_for_rebuild = np.ceil(total_rebuild_time_s / (rebuild_window_min * 60))

print(f"  New images/day:    {NEW_IMAGES_PER_DAY:>10,}")
print(f"  New images/sec:    {new_img_per_sec:>10.1f}")
print(f"  GPUs for incremental updates: {gpus_incremental:.0f}")
print(f"  GPUs for 30-min nightly rebuild: {gpus_for_rebuild:.0f} A100s")

print("\n4. TOTAL INFRASTRUCTURE ESTIMATE")
print("-" * 45)
total_gpu_cost_per_month = gpus_for_rebuild * 3  # approx 3 GPUs × ~$2/hr
print(f"  Embedding servers (PQ): {n_servers_pq:.0f} × 512GB RAM machines")
print(f"  ANN search servers:     {n_ann_servers*3:.0f} (3x replicated)")
print(f"  GPU servers (inference): ~{gpus_for_rebuild:.0f} A100 GPUs (nightly batch)")
print(f"  GPU servers (incremental): {gpus_incremental:.0f} A100 GPU (continuous)")

# ---- Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Storage comparison
ax = axes[0]
sizes = [raw_tb, fp16_tb, pq_tb]
labels = ['Float32\n(204.8 TB)', 'Float16\n(102.4 TB)', 'PQ32x8\n(6.4 TB)']
colors = ['#E53935', '#FF9800', '#4CAF50']
bars = ax.bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Storage (TB)', fontsize=13)
ax.set_title('Embedding Storage\nat 200B Images', fontsize=13, fontweight='bold')
ax.set_yscale('log')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1,
            f'{size:.0f} TB', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# QPS breakdown
ax2 = axes[1]
ax2.bar(['Avg QPS', 'Peak QPS (5x)'], [avg_qps, peak_qps],
        color=['#1E88E5', '#E53935'], edgecolor='white', linewidth=1.5)
ax2.set_ylabel('Queries Per Second', fontsize=13)
ax2.set_title('Query Load\n(1B queries/day)', fontsize=13, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.grid(True, alpha=0.3, axis='y')

# GPU requirements
ax3 = axes[2]
gpu_scenarios = [gpus_for_rebuild, gpus_incremental, 50]
gpu_labels = [f'Nightly\nRebuild\n({gpus_for_rebuild:.0f})', 
              f'Incremental\nUpdates\n({gpus_incremental:.0f})',
              'Inference\nBuffer\n(~50)']
ax3.bar(gpu_labels, gpu_scenarios, color=['#9C27B0', '#4CAF50', '#FF9800'],
        edgecolor='white', linewidth=1.5)
ax3.set_ylabel('A100 GPUs Required', fontsize=13)
ax3.set_title('GPU Infrastructure\n(A100, ResNet-50)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('200B Image Scale: Infrastructure Requirements', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Interview Cheat Sheet: ANN Search & Serving

### 30-Second Pitch

> "For serving at 200 billion images, we use a two-pipeline architecture. The offline indexing pipeline batch-processes all images through a GPU fleet to generate embeddings, then builds a FAISS IVFFlat index with 65,536 clusters. The online prediction pipeline preprocesses the query image, generates its embedding in real-time, runs ANN search with nprobe=128 for 95% recall in under 10ms, and passes the top-500 candidates to a re-ranking service that applies business rules — privacy filters, deduplication, content policy, and diversity constraints — before returning the final top-50 results. Product quantization reduces storage from 200 TB to 6 TB, making this economically viable."

### Key Phrases to Drop

| Phrase | Context |
|--------|---------|
| "O(N×D) for exact, O(D log N) for ANN" | Comparing brute force vs. ANN |
| "IVF with nlist=sqrt(N) and nprobe controls the recall-speed knob" | Explaining IVF |
| "FAISS for CPU/GPU, ScaNN for TPU-optimized workloads" | Library choices |
| "Product quantization reduces memory 32× with minimal recall loss" | Memory optimization |
| "Re-ranking applies privacy, dedup, policy, and diversity constraints" | Serving details |
| "Eventually consistent index — new images appear within minutes" | Consistency model |
| "nightly index rebuild + incremental updates for fresh content" | Indexing strategy |

### Complexity Reference Card

| Operation | Complexity | Notes |
|-----------|-----------|-------|
| Exact NN | O(N × D) | Unusable at N > 10M |
| KD-tree (low-D) | O(D log N) | Fails at D > 20 |
| LSH search | O(bucket_size × D) | Tunable recall |
| IVF search | O(nprobe × N/nlist × D) | Main workhorse |
| HNSW search | O(log N × D) | Best latency, high memory |
| PQ encode | O(M × k* × D) | One-time per vector |

### Common Follow-up Questions

| Question | Strong Answer |
|----------|---------------|
| How do you handle model updates? | Rebuild the entire index nightly; serve old index until new one is ready. Zero-downtime swap. |
| What if 95% recall is not enough? | Increase nprobe, switch from IVFFlat to HNSW for better recall-latency tradeoff. |
| How do you monitor search quality in production? | Track CTR, nDCG via sample, embedding distribution drift. Alert on recall degradation. |
| What about cold-start for new images? | Incremental indexing ensures new images are searchable within minutes. No need to wait for full rebuild. |
| Why not use a vector database (Pinecone, Weaviate)? | Valid for smaller scale. At 200B we need custom FAISS with PQ for memory efficiency and cost. |

---

*Next notebook: Complete 45-minute mock interview walkthrough with scripts, scoring rubrics, and power moves.*

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)